In [21]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# noise = 0 clean and clearly shaped data
# noise = 0.10 slightly scattered points
# noise = 0.30 more scattered points, make classification harder
X, y = make_moons(
    n_samples=1200,
    noise=0.30,
    random_state=42
)

y = y.reshape(-1, 1)

In [22]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print("Training Examples : ", len(X_train))
print("Validation Examples : ", len(X_val))
print("Test Examples : ", len(X_test))

Training Examples :  840
Validation Examples :  180
Test Examples :  180


In [23]:
np.random.seed(42)

input_size = 2 
hidden_size = 16
output_size = 1

W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(1 / hidden_size)
b2 = np.zeros((1, output_size))

learning_rate = 0.03
epochs = 500
batch_size = 32

patience = 20
min_delta = 1e-4

In [24]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def forward(X):
    z1 = X @ W1 + b1
    a1 = np.maximum(0, z1)

    z2 = a1 @ W2 + b2
    probabilities = sigmoid(z2)

    return z1, a1, probabilities

def binary_cross_entropy(y_true, probabilities):
    probabilities = np.clip(probabilities, 1e-12, 1 - 1e-12)
    return -np.mean(y_true * np.log(probabilities) + (1 - y_true) * np.log(1 - probabilities))

def calculate_loss(X, y):
    _, _, probabilities = forward(X)
    return binary_cross_entropy(y, probabilities)

def accuracy(X, y):
    _, _, probabilities = forward(X)
    predictions = (probabilities >= 0.5).astype(int)
    return np.mean(predictions == y)

In [25]:
best_val_loss = float("inf")
best_epoch = 0
best_parameters = None
epochs_without_improvement = 0

for epoch in range(1, epochs + 1):
    indices = np.random.permutation(len(X_train))
    X_shuffled = X_train[indices]
    y_shuffled = y_train[indices]

    for start in range(0, len(X_train), batch_size):
        end = start + batch_size

        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]
        m = len(X_batch)

        z1, a1, probabilities = forward(X_batch)
        dz2 = (probabilities - y_batch) / m

        dW2 = a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2 @ W2.T
        dz1 = da1 * (z1 > 0)

        dW1 = X_batch.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
    
    train_loss = calculate_loss(X_train, y_train)
    val_loss = calculate_loss(X_val, y_val)

    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        best_epoch = epoch

        best_parameters = (W1.copy(), b1.copy(), W2.copy(), b2.copy())

        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 25 == 0:
        print(
            f"Epoch : {epoch:03d} | "
            f"Train Loss : {train_loss:.4f} | "
            f"Validation Loss : {val_loss:.4f} | "
        )

    if epochs_without_improvement >= patience:
        print(f"\nEarly Stopping At Epoch : {epoch}.")
        break

W1, b1, W2, b2 = best_parameters

print("\nBest Epoch : ", best_epoch)
print(f"Best Validation Loss : {best_val_loss:.4f}")

test_loss = calculate_loss(X_test, y_test)
test_accuracy = accuracy(X_test, y_test)

print("\nTest Loss : ", test_loss)
print("Test Accuracy : ", test_accuracy)

Epoch : 001 | Train Loss : 0.4643 | Validation Loss : 0.4723 | 
Epoch : 025 | Train Loss : 0.3386 | Validation Loss : 0.3248 | 
Epoch : 050 | Train Loss : 0.3324 | Validation Loss : 0.3156 | 
Epoch : 075 | Train Loss : 0.3235 | Validation Loss : 0.3029 | 
Epoch : 100 | Train Loss : 0.3119 | Validation Loss : 0.2892 | 
Epoch : 125 | Train Loss : 0.2994 | Validation Loss : 0.2741 | 
Epoch : 150 | Train Loss : 0.2873 | Validation Loss : 0.2592 | 
Epoch : 175 | Train Loss : 0.2765 | Validation Loss : 0.2443 | 
Epoch : 200 | Train Loss : 0.2676 | Validation Loss : 0.2319 | 
Epoch : 225 | Train Loss : 0.2609 | Validation Loss : 0.2232 | 
Epoch : 250 | Train Loss : 0.2559 | Validation Loss : 0.2154 | 
Epoch : 275 | Train Loss : 0.2521 | Validation Loss : 0.2098 | 
Epoch : 300 | Train Loss : 0.2492 | Validation Loss : 0.2047 | 
Epoch : 325 | Train Loss : 0.2471 | Validation Loss : 0.2009 | 
Epoch : 350 | Train Loss : 0.2455 | Validation Loss : 0.1976 | 
Epoch : 375 | Train Loss : 0.2442 | Vali